# 01 EDA (member A)

Goal: understand the data and find the problems to fix. Every figure is saved to `reports/figures/` for the report and slides.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import FIGURE_DIR, TARGET
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [2]:
from src.data import load_data
train, test = load_data()
print(train.shape, test.shape)
train.head()

(1460, 81) (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 1. Target: SalePrice
Right-skewed -> close to normal after log (iter1 uses log; control: iter1_no_log).

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train[TARGET], kde=True, ax=ax[0]).set_title(f"SalePrice (skew={train[TARGET].skew():.2f})")
sns.histplot(np.log1p(train[TARGET]), kde=True, ax=ax[1]).set_title(f"log1p(SalePrice) (skew={np.log1p(train[TARGET]).skew():.2f})")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "target_distribution.png", dpi=150)

## 2. Features most correlated with price

In [4]:
corr = train.select_dtypes("number").corr()[TARGET].drop(TARGET).sort_values(ascending=False)
top = corr.head(10)
print(top)
plt.figure(figsize=(8, 4)); sns.barplot(x=top.values, y=top.index)
plt.title("Top 10 correlation with SalePrice"); plt.tight_layout()
plt.savefig(FIGURE_DIR / "top_correlations.png", dpi=150)

OverallQual     0.790982
GrLivArea       0.708624
GarageCars      0.640409
GarageArea      0.623431
TotalBsmtSF     0.613581
1stFlrSF        0.605852
FullBath        0.560664
TotRmsAbvGrd    0.533723
YearBuilt       0.522897
YearRemodAdd    0.507101
Name: SalePrice, dtype: float64


In [5]:
cols = [TARGET] + list(top.index[:6])
plt.figure(figsize=(9, 7)); sns.heatmap(train[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "corr_heatmap.png", dpi=150)

## 3. Outliers
Bottom right: huge living area but low price (removed in iter2).

In [6]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=train, x="GrLivArea", y=TARGET)
out = train[(train.GrLivArea > 4000) & (train[TARGET] < 300000)]
plt.scatter(out.GrLivArea, out[TARGET], color="red", s=80, label="outliers"); plt.legend()
plt.tight_layout(); plt.savefig(FIGURE_DIR / "outliers.png", dpi=150)
out[["Id", "GrLivArea", TARGET]]

,Id,GrLivArea,SalePrice
523,524,4676,184750
1298,1299,5642,160000


## 4. Missing values
Note: many NAs mean "no such facility", see `data_description.txt` (handled by fill_missing in iter2).

In [7]:
full = pd.concat([train.drop(columns=TARGET), test])
missing = (full.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]
print(f"{len(missing)} columns have missing values")
shown = missing[missing >= 0.5]  # the rest are only a handful of rows each
plt.figure(figsize=(8, 6)); ax = sns.barplot(x=shown.values, y=shown.index)
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3, fontsize=8)
ax.set(xlabel="% missing (train + test)", ylabel="", title=f"Columns with >= 0.5% missing ({len(shown)} of {len(missing)})")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "missing_values.png", dpi=150)
missing

34 columns have missing values


PoolQC          99.657417
MiscFeature     96.402878
Alley           93.216855
Fence           80.438506
MasVnrType      60.500171
FireplaceQu     48.646797
LotFrontage     16.649538
GarageYrBlt      5.447071
GarageCond       5.447071
GarageFinish     5.447071
GarageQual       5.447071
GarageType       5.378554
BsmtCond         2.809181
BsmtExposure     2.809181
BsmtQual         2.774923
BsmtFinType2     2.740665
BsmtFinType1     2.706406
MasVnrArea       0.787941
MSZoning         0.137033
BsmtHalfBath     0.068517
Utilities        0.068517
Functional       0.068517
BsmtFullBath     0.068517
BsmtFinSF1       0.034258
Exterior1st      0.034258
TotalBsmtSF      0.034258
BsmtUnfSF        0.034258
BsmtFinSF2       0.034258
SaleType         0.034258
KitchenQual      0.034258
GarageCars       0.034258
GarageArea       0.034258
Electrical       0.034258
Exterior2nd      0.034258
dtype: float64

## 5. Categorical features: e.g. Neighborhood / OverallQual

In [8]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
order = train.groupby("Neighborhood")[TARGET].median().sort_values().index
sns.boxplot(data=train, x="Neighborhood", y=TARGET, order=order, ax=ax[0]); ax[0].tick_params(axis="x", rotation=90)
sns.boxplot(data=train, x="OverallQual", y=TARGET, ax=ax[1])
plt.tight_layout(); plt.savefig(FIGURE_DIR / "categorical_vs_price.png", dpi=150)

## Findings (for report section 2)

- **Data size:** 1,460 training houses and 1,459 test houses, 79 features (36 numeric, 43 categorical).
- **Target is right-skewed:** SalePrice skew = 1.88 (median $163,000, mean $180,921). After `log1p` the skew drops to 0.12, close to normal. The competition metric is also measured on log prices, so the baseline already trains on log(SalePrice).
- **Quality and size drive price:** the strongest correlations are OverallQual (0.79), GrLivArea (0.71), GarageCars (0.64), GarageArea (0.62) and TotalBsmtSF (0.61). Median price rises steadily from $50k at OverallQual 1 to $432k at OverallQual 10.
- **Location matters:** median price by Neighborhood ranges from $88k (MeadowV) to $315k (NridgHt), so Neighborhood is worth keeping as a category.
- **Two clear outliers:** Id 524 and 1299 have GrLivArea > 4,000 sq ft but sold for under $200k (both Partial sales in Edwards). They break the size-price trend. -> removed in Stage 2.
- **Missing values are mostly "no facility":** 34 columns have NAs. PoolQC (99.7%), MiscFeature (96.4%), Alley (93.2%), Fence (80.4%), FireplaceQu (48.6%) and the Garage*/Bsmt* columns are NA because the house has no pool/alley/fence/fireplace/garage/basement (see `data_description.txt`). They should be filled with "None" (or 0 for areas/counts), not with the median. LotFrontage (16.6%) is truly missing -> fill by Neighborhood median. -> Stage 2.

## Stage 1 results

| Version | CV RMSE (log) | Per-fold | Kaggle |
|---|---|---|---|
| `iter1_no_log` (control, raw SalePrice) | 0.15389 ± 0.01588 | 0.157, 0.149, **0.180**, 0.153, 0.130 | not submitted |
| `iter1_baseline` (log SalePrice) | 0.14644 ± 0.03923 | 0.136, 0.129, **0.224**, 0.127, 0.116 | **0.13241** |

- The log transform improves 4 of 5 folds by about 0.02.
- The bold fold is the one that contains the 2 outliers. With log it jumps to 0.224 (vs ~0.13 elsewhere), which is why the std is high. This is the main motivation for removing outliers in Stage 2.


# Stage 2: data cleaning

Cleaning code: `remove_outliers` in `src/data.py` and `fill_missing` in `src/features.py`.

Rules:
1. **Outliers:** drop the 2 houses with GrLivArea > 4,000 and SalePrice < $300k (Id 524, 1299).
2. **Facility present but a field is NA** (e.g. MasVnrArea > 0 but MasVnrType NA): truly missing -> most common value among houses that have the facility.
3. **NA = no facility** (pool, alley, fence, fireplace, garage, basement, veneer, misc): -> `"None"` for categories, `0` for areas/counts.
4. **GarageYrBlt:** fix the test-set typo 2207 -> remodel year 2007; houses without a garage get the house's YearBuilt instead of 0.
5. **LotFrontage:** median of the same Neighborhood.
6. **Anything left** (a few rows in MSZoning, Utilities, KitchenQual, ...): mode / median.

In [9]:
from src.data import remove_outliers
from src.features import fix_types, fill_missing

full = pd.concat([train.drop(columns=TARGET), test], ignore_index=True)
cleaned = fill_missing(fix_types(full))
print("NA before:", int(full.isna().sum().sum()), "-> after:", int(cleaned.isna().sum().sum()))

# Rows where the facility exists but a field was NA (rule 2)
check = {"MasVnrType": full.MasVnrType.isna() & (full.MasVnrArea > 0),
         "PoolQC": full.PoolQC.isna() & (full.PoolArea > 0),
         "BsmtQual": full.BsmtQual.isna() & (full.TotalBsmtSF > 0),
         "GarageFinish": full.GarageFinish.isna() & (full.GarageArea > 0)}
for col, mask in check.items():
    print(f"{col:<13} {mask.sum()} rows -> {cleaned.loc[mask, col].unique().tolist()}")
print("GarageYrBlt range:", cleaned.GarageYrBlt.min(), "-", cleaned.GarageYrBlt.max())

NA before: 15707 -> after: 0
MasVnrType    8 rows -> ['BrkFace']
PoolQC        3 rows -> ['Ex']
BsmtQual      2 rows -> ['TA']
GarageFinish  1 rows -> ['Unf']
GarageYrBlt range: 1872.0 - 2010.0


## Confirming the outliers with CV residuals

EDA showed 2 suspicious points in the GrLivArea vs SalePrice plot. Before removing them, check whether they actually hurt the model:
run 5-fold CV on the cleaned data **without** removing anything (`iter2b_missing_only`), so every house is predicted exactly once by a model that never saw it, then rank the houses by prediction error.

In [10]:
from src.pipeline import score_cv
from src.versions import VERSIONS

s = score_cv(VERSIONS["iter2b_missing_only"], (train, test))
res = pd.DataFrame({
    "Id": train.Id, "actual": train[TARGET], "predicted": s["oof"].round(),
    "log_error": np.log(s["oof"]) - np.log(train[TARGET]),
    "GrLivArea": train.GrLivArea, "SaleCondition": train.SaleCondition, "Neighborhood": train.Neighborhood,
})
res["abs_error"] = res.log_error.abs()
print("Fold scores:", [round(x, 3) for x in s["fold_scores"]])
print("Median absolute log error:", round(res.abs_error.median(), 3))
res.sort_values("abs_error", ascending=False).head(10)

Fold scores: [0.138, 0.129, 0.222, 0.128, 0.113]
Median absolute log error: 0.063


,Id,actual,predicted,log_error,GrLivArea,SaleCondition,Neighborhood,abs_error
1298,1299,160000,2747434.0,2.843249,5642,Partial,Edwards,2.843249
523,524,184750,1031326.0,1.719597,4676,Partial,Edwards,1.719597
30,31,40000,88691.0,0.796281,1317,Normal,IDOTRR,0.796281
632,633,82500,178765.0,0.773272,1411,Family,NWAmes,0.773272
462,463,62383,125160.0,0.696300,864,Normal,Sawyer,0.696300
968,969,37900,75954.0,0.695174,968,Abnorml,OldTown,0.695174
916,917,35311,68232.0,0.658715,480,Abnorml,IDOTRR,0.658715
1324,1325,147000,281098.0,0.648272,1795,Partial,Somerst,0.648272
495,496,34900,65973.0,0.636755,720,Abnorml,IDOTRR,0.636755
1423,1424,274970,146005.0,-0.633021,2201,Alloca,Edwards,0.633021


In [11]:
sq = res.log_error ** 2
top2 = sq.nlargest(2)
print(f"The 2 worst houses ({', '.join(map(str, train.Id[top2.index]))}) are "
      f"{len(top2) / len(sq):.2%} of the data but cause {top2.sum() / sq.sum():.1%} of the total squared error")

is_out = res.Id.isin([524, 1299])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(res.actual[~is_out], res.predicted[~is_out], s=8, alpha=0.5, label="other houses")
ax[0].scatter(res.actual[is_out], res.predicted[is_out], s=80, color="red", label="Id 524, 1299")
lim = [3e4, 3e6]; ax[0].plot(lim, lim, "k--", lw=1, label="perfect prediction")
ax[0].set(xscale="log", yscale="log", xlabel="Actual SalePrice", ylabel="CV predicted SalePrice",
          title="Out-of-fold predictions (no outlier removal)"); ax[0].legend()
ax[1].hist(res.abs_error[~is_out], bins=60, label="other houses")
for v in res.abs_error[is_out]:
    ax[1].axvline(v, color="red", lw=2)
ax[1].set(xlabel="|log error|", ylabel="Houses", title="Error distribution (red lines = Id 524, 1299)")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "cv_residuals.png", dpi=150)

The 2 worst houses (1299, 524) are 0.14% of the data but cause 33.1% of the total squared error


**Result:** the CV model predicts about **$2.7M for Id 1299** (sold for $160k) and **$1.0M for Id 524** (sold for $185k). Their log errors (2.84 and 1.72) are more than twice the next-worst house (0.80), while the typical error is around 0.06. These 2 houses are 0.14% of the data but cause **33% of the total squared error**.

**Why they are so far off:** both are the newest, largest, highest-quality (OverallQual 10) houses in the data, yet they sold for about $30-40 per sq ft, compared with a median of $174 per sq ft for quality-10 houses. Two similar 4,300-4,500 sq ft quality-10 houses in NoRidge sold for $745k-755k. Both are SaleType `New` with SaleCondition `Partial` ("home was not completed when last assessed"), sold in the year they were built, and both are in Edwards, a low-priced neighborhood where no other house is larger than 2,201 sq ft. The recorded prices most likely do not reflect normal market sales of finished houses.

**Decision:** EDA flagged them, CV residuals confirm they distort the model, and there is a plausible reason why their prices are not representative -> remove them from training.

In [12]:
clean_train = remove_outliers(train)
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.scatterplot(data=train, x="GrLivArea", y=TARGET, ax=ax[0]).set_title(f"Before ({len(train)} houses)")
sns.scatterplot(data=clean_train, x="GrLivArea", y=TARGET, ax=ax[1]).set_title(f"After removing outliers ({len(clean_train)} houses)")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "outliers_before_after.png", dpi=150)

## Stage 2 results

All CV numbers use **fair CV**: the same 5 folds over all 1,460 houses for every version. Outlier removal only changes which houses a model is trained on; every version is scored on the same houses (see "Lesson" below for why).

| Version | Change vs iter1 | CV RMSE (all houses) | Per-fold | CV (normal houses only) | Kaggle |
|---|---|---|---|---|---|
| `iter1_baseline` | - | 0.14644 ± 0.03923 | 0.136, 0.129, **0.224**, 0.127, 0.116 | 0.12309 | 0.13241 |
| `iter2b_missing_only` | missing values only | 0.14615 ± 0.03875 | 0.138, 0.129, **0.222**, 0.128, 0.113 | 0.12324 | not submitted |
| `iter2a_outliers_only` | outliers only | 0.13722 ± 0.04388 | 0.127, 0.112, **0.224**, 0.118, 0.106 | 0.11387 | not submitted |
| `iter2_cleaned` | both | 0.13696 ± 0.04336 | 0.130, 0.111, **0.222**, 0.117, 0.104 | 0.11405 | 0.13281 |

- **Removing the 2 outliers from training improves the other 4 folds** by about 0.01 each (e.g. 0.129 -> 0.111). Without the outliers pulling the fit down for large houses, the model predicts normal houses better: 0.1231 -> 0.1140 on the 1,458 normal houses.
- **Fold 3 stays at ~0.22** because it is the fold that contains both outliers. They are always in its validation part, so no version ever trains on them there, and they are still predicted at $1.0M-2.8M. No model can predict these 2 houses well, which is the point.
- **Missing-value handling barely changes the score for Ridge** (0.1231 -> 0.1232 on normal houses). Its value is correctness: "no garage" and "unknown garage" are different things, and Stage 3 features (e.g. garage/house age) need sensible values instead of 0 or NaN.
- The refined rules (2) and (4) changed CV by only ~0.0002 compared with a plain "NA -> None/0" fill, which is within noise. They are kept because they are more correct and avoid a fake "garage built in year 0".
- **Kaggle did not move** (0.13241 -> 0.13281, +0.0004, within noise). A likely reason: test Id 2550 is a 5,095 sq ft, quality-10, new Partial sale in Edwards, the same pattern as the 2 removed outliers. A model trained without them predicts an even higher price for 2550, and if it also sold cheaply, that single house can cancel the gain on normal houses. We cannot check this because the test labels are hidden.

### Lesson: compare CV on the same houses

Our first CV removed the outliers *before* splitting into folds. iter2 then scored 0.11477 vs iter1's 0.14644, a 0.032 "improvement". But iter2 was simply no longer being tested on the 2 hardest houses: the two numbers were measured on different houses. Kaggle showed no improvement, which made us look into it. With fair CV (same validation houses for every version) the real gain is ~0.009, and fair CV is now built into `src/pipeline.py` for all versions.

**Decision:** keep the outlier removal for Stage 3 and 4. On a like-for-like comparison it improves predictions for normal houses, and the Kaggle change is within noise.

### Notes for Stage 3 (handoff to B)
- Compare iter3 against `iter2_cleaned`: CV **0.13696** (all houses) / **0.11405** (normal houses). `cv_rmse_normal` in the log is less dominated by the 2 outliers, so it shows small feature effects more clearly.
- Id 2550 (test) has YrSold < YearBuilt and Id 1877 (test) has YearRemodAdd < YearBuilt, so `HouseAge` / `RemodAge` can be negative for these rows; consider clipping at 0.